# DSA Assignment 6 – Food Delivery Time Prediction

Complete notebook with preprocessing, clustering, and neural network modeling.

## Phase 1 – Data Preprocessing & Feature Engineering

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("Food_Delivery_Time_Prediction.csv")
df.head()

In [ ]:
# Handle missing values
df = df.dropna()

In [ ]:
# Encode categorical variables
le = LabelEncoder()
categorical_cols = ['Weather', 'Traffic', 'Order_Priority']

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

In [ ]:
# Haversine distance calculation
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

df['Distance'] = df.apply(lambda row: haversine(
    row['Restaurant_Lat'],
    row['Restaurant_Lon'],
    row['Customer_Lat'],
    row['Customer_Lon']
), axis=1)

In [ ]:
# Rush hour feature
def rush_hour(hour):
    return 1 if (7 <= hour <= 10 or 17 <= hour <= 21) else 0

df['Rush_Hour'] = df['Order_Hour'].apply(rush_hour)

In [ ]:
# Normalize
scaler = StandardScaler()
df[['Distance']] = scaler.fit_transform(df[['Distance']])

## Phase 2 – Clustering

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

X = df[['Distance', 'Traffic', 'Weather']]

wcss = []
for i in range(1, 10):
    kmeans = KMeans(n_clusters=i)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)

plt.plot(range(1, 10), wcss)
plt.title("Elbow Method")
plt.xlabel("Clusters")
plt.ylabel("WCSS")
plt.show()

In [ ]:
# K-Means clustering
kmeans = KMeans(n_clusters=3)
df['Cluster'] = kmeans.fit_predict(X)

plt.scatter(df['Distance'], df['Traffic'], c=df['Cluster'])
plt.xlabel("Distance")
plt.ylabel("Traffic")
plt.title("K-Means Clusters")
plt.show()

In [ ]:
# Hierarchical clustering
import scipy.cluster.hierarchy as sch

plt.figure(figsize=(10,5))
dendrogram = sch.dendrogram(sch.linkage(X, method='ward'))
plt.title("Dendrogram")
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering

hc = AgglomerativeClustering(n_clusters=3)
df['H_Cluster'] = hc.fit_predict(X)

## Phase 3 – Neural Network

In [ ]:
X = df[['Distance', 'Traffic', 'Weather', 'Rush_Hour']]
y = df['Delivery_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()
model.add(Dense(16, activation='relu', input_dim=X.shape[1]))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(X_train, y_train, epochs=20, batch_size=10)

In [ ]:
from sklearn.metrics import classification_report

y_pred = (model.predict(X_test) > 0.5)
print(classification_report(y_test, y_pred))

## Bonus – Logistic Regression Comparison

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

print("Logistic Regression Accuracy:", lr.score(X_test, y_test))